# A1 独立评估

同类型加权 1-WD、细胞均值 MSE、区域均值 Pearson；同时绘制文献区域的边缘分布和按类型分布。

使用 simscDNAm 环境的 Jupyter，修改 `evaluate/config.yaml` 后从上到下运行。本入口只执行本实验，结果写入配置输出目录的 `A1/`，图写入其 `figures/`。

In [ ]:
from pathlib import Path
PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "model/src/model.py").is_file())
CONFIG_PATH = PROJECT / "evaluate/config.yaml"
EXPERIMENT = "A1"

## 读取配置与指标

相对路径以配置文件所在目录为准；各实验使用同一随机种子和同一批抽样细胞。

In [ ]:
from contextlib import ExitStack
import importlib.metadata
import json
import sys

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_limits
import yaml

PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "model/src/model.py").is_file())
sys.path.insert(0, str(PROJECT))
from data.evaluation import (dense_block, read_labels, region_pairs, validate_real_duplicates,
    natural_reference_statistics, select_features, imputed_features, validate_region_axis, sample_rows)
if Path(sys.prefix).name.lower() != "simscdnam":
    raise RuntimeError(f"请切换到 simscDNAm 的 Jupyter 内核，当前为 {sys.executable}")
CONFIG_PATH = CONFIG_PATH.expanduser().resolve()
config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
if not isinstance(config, dict):
    raise ValueError("配置必须是 YAML 映射")
required = {"real_h5ad", "synthetic_h5ad", "output_dir"}
if not required.issubset(config) or not isinstance(config["synthetic_h5ad"], dict) or not config["synthetic_h5ad"]:
    raise ValueError("请填写 real_h5ad、非空 synthetic_h5ad 方法映射和 output_dir")
defaults = {"reference_h5ad": None, "region_pairs_csv": None, "label_key": "Subclass",
    "duplicate_count": 1000, "seed": 42, "evaluation_cells": None, "a3_top_regions": 50000,
    "pca_components": 50, "milisi_neighbors": 30, "umap_seed": 2022, "spearman_top_regions": 2000,
    "a1_regions_csv": None, "genome_assembly": None, "read_in_memory": False}
config = {**defaults, **config}
if not isinstance(config["read_in_memory"], bool):
    raise ValueError("read_in_memory必须为布尔值")
if config["label_key"] != "Subclass":
    raise ValueError("本流程统一使用真实类型 Subclass，请将真实标签字段改名为 Subclass")
if config["duplicate_count"] != 1000:
    raise ValueError("本流程固定最后 1000 个区间为复制区间")
for key in ("a3_top_regions", "pca_components", "milisi_neighbors", "spearman_top_regions"):
    if not isinstance(config[key], int) or isinstance(config[key], bool) or config[key] < 2:
        raise ValueError(f"{key} 必须为不小于 2 的整数")
if config["evaluation_cells"] is not None and (not isinstance(config["evaluation_cells"], int)
        or isinstance(config["evaluation_cells"], bool) or config["evaluation_cells"] < 3):
    raise ValueError("evaluation_cells 必须为 null 或不小于 3 的整数")
if any(not isinstance(name, str) or not name.strip() or name in {"真实数据", "训练参照", "CVAEKMeansAligned"}
       for name in config["synthetic_h5ad"]):
    raise ValueError("方法名须为非空字符串，且不能使用保留名称或暂不参与的 CVAEKMeansAligned")

def resolve_path(value):
    path = Path(value).expanduser()
    return (CONFIG_PATH.parent / path).resolve() if not path.is_absolute() else path.resolve()

plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans", "pdf.fonttype": 42,
                     "axes.spines.top": False, "axes.spines.right": False})
print("Jupyter 内核：", sys.executable)
print("配置文件：", CONFIG_PATH)
from evaluate.A1.metrics import a1_metrics


## 本实验绘图

图中文字使用英文，缺失值和超出 0–1 的值按原协议处理。

In [ ]:
from matplotlib.colors import to_rgba
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator


PLOT_COLORS = {"真实数据": "#A6A6A6", "Real": "#A6A6A6",
    "simscDNAm": "#3B73B9", "CVAE_training_mask": "#3B73B9", "FittedMarginal": "#269D91",
    "simATAC": "#A487C9", "simCAS": "#D95F02", "STADiffuser": "#B44C76"}


def plot_name(name):
    return {"真实数据": "Real", "CVAE_training_mask": "CVAE (training mask)"}.get(name, name)


def plot_color(name):
    return PLOT_COLORS.get(name, "#3B73B9")


def style_axis(axis):
    axis.spines[["top", "right"]].set_visible(False)
    axis.spines[["left", "bottom"]].set_linewidth(0.7)
    axis.grid(axis="y", color="#E8E8E8", linewidth=0.5)
    axis.set_axisbelow(True)


def save_figure(figure, output, stem):
    with plt.rc_context({"pdf.fonttype": 42}):
        for extension in ("png", "pdf"):
            figure.savefig(output / "figures" / f"{stem}.{extension}", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(figure)


def probability_histogram(values, edges):
    """按全部有限原值归一化；越界值单独报告，不压入边缘分箱。"""
    finite = np.asarray(values)[np.isfinite(values)]
    probabilities = np.histogram(finite, bins=edges)[0].astype(float)
    outside = np.nan
    if finite.size:
        probabilities /= finite.size
        outside = np.mean((finite < edges[0]) | (finite > edges[-1]))
    return probabilities, outside, int(finite.size)


def load_plot_regions(real, natural_count, path, assembly):
    """显式读取展示证据清单；不把随机区间或复制尾部当作文献区域。"""
    table = pd.read_csv(path)
    required = {"原region", "展示名称", "基因组版本", "文献来源", "单区域展示", "展示细胞类型"}
    if not required.issubset(table) or table.empty:
        raise ValueError("展示区间清单缺少区域、名称、基因组版本、文献来源或单区域展示信息")
    if table[["原region", "展示名称", "基因组版本", "文献来源"]].isna().any().any():
        raise ValueError("展示区间及文献证据不能为空")
    if not assembly or not table["基因组版本"].eq(assembly).all():
        raise ValueError("展示区间清单的基因组版本与配置不一致")
    indices = real.var_names[:natural_count].get_indexer(table["原region"])
    if np.any(indices < 0) or table["原region"].duplicated().any():
        raise ValueError("展示区间须为互不重复且存在的原始区间，不能使用复制尾部")
    flags = pd.to_numeric(table["单区域展示"], errors="coerce")
    if not flags.isin([0, 1]).all() or flags.sum() != 1:
        raise ValueError("清单须且仅须指定一个单区域展示窗口")
    detail = table.loc[flags.eq(1)].iloc[0]
    levels = str(detail["展示细胞类型"]).split("|")
    if pd.isna(detail["展示细胞类型"]) or any(not level.strip() for level in levels) or len(set(levels)) != len(levels):
        raise ValueError("单区域展示须提供以 | 分隔的不重复真实细胞类型")
    table["单区域展示"] = flags.astype(int)
    table.insert(0, "原region索引", indices)
    return table


def marginal_legend(figure, methods, y=0.99):
    handles = [Patch(facecolor=to_rgba(plot_color(name), 0.4), edgecolor=plot_color(name), label=plot_name(name))
               for name in ["真实数据", *methods]]
    figure.legend(handles=handles, loc="upper center", ncol=len(handles), frameon=False, fontsize=8,
                  bbox_to_anchor=(0.5, y))


def draw_marginal(axis, real_histogram, synthetic_histogram, method, edges):
    style_axis(axis)
    for name, histogram, edge_color in [("真实数据", real_histogram, "#555555"),
                                       (method, synthetic_histogram, plot_color(method))]:
        axis.stairs(histogram[0], edges, fill=True, linewidth=0.8,
                    edgecolor=edge_color, facecolor=to_rgba(plot_color(name), 0.35))
    outside = synthetic_histogram[1]
    annotation = f"Out of [0, 1]: {outside:.1%}" if np.isfinite(outside) else "Out of [0, 1]: n/a"
    axis.text(0.98, 0.95, annotation, transform=axis.transAxes, fontsize=6.5, va="top", ha="right")
    axis.set(xlim=(0, 1), xticks=np.linspace(0, 1, 6))
    axis.yaxis.set_major_locator(MaxNLocator(3))
    axis.tick_params(labelsize=7, length=3)


def plot_marginal_regions(datasets, rows, methods, regions, output):
    selected = regions["原region索引"].to_numpy(dtype=int)
    regions.to_csv(output / "A1_plot_regions.csv", index=False)
    values = {name: dense_block(datasets[name], rows[name], selected) for name in ["真实数据", *methods]}
    edges = np.linspace(0, 1, 31)
    figure, axes = plt.subplots(len(selected), len(methods), figsize=(3.15 * len(methods), 2.0 * len(selected) + 0.7),
                                sharex=True, sharey="row", squeeze=False)
    figure.subplots_adjust(left=0.075, right=0.995, bottom=0.85 / figure.get_figheight(), top=1 - 0.95 / figure.get_figheight(), wspace=0.10, hspace=0.34)
    source_tables = []
    for i, row in regions.iterrows():
        histograms = {name: probability_histogram(values[name][:, i], edges) for name in ["真实数据", *methods]}
        upper = max(0.1, np.ceil(max(item[0].max() for item in histograms.values()) * 12) / 10)
        for name, (probabilities, outside, count) in histograms.items():
            source_tables.append(pd.DataFrame({"原region": row["原region"], "方法": name,
                "分箱左界": edges[:-1], "分箱右界": edges[1:], "概率": probabilities,
                "有限值数": count, "越界比例": outside}))
        for j, method in enumerate(methods):
            axis = axes[i, j]
            draw_marginal(axis, histograms["真实数据"], histograms[method], method, edges)
            axis.text(0.02, 0.95, str(row["原region"]), transform=axis.transAxes, fontsize=6, va="top", color="#555555")
            axis.set_ylim(0, upper)
            if i == 0:
                axis.set_title(plot_name(method), color=plot_color(method), fontsize=10, weight="bold", pad=7)
            if i == len(selected) - 1:
                axis.set_xlabel("Methylation level", fontsize=8)
            if j == 0:
                axis.set_ylabel(f"{row['展示名称']}\nProbability per bin", fontsize=8)
    marginal_legend(figure, methods, 0.98)
    figure.suptitle("Literature-guided Tle4 locus windows (mm10)" if regions["展示名称"].str.startswith("Tle4").all()
                     else "Literature-guided marginal distributions", fontsize=11, weight="bold", y=1.01)
    figure.text(0.075, 0.015, "All cell types pooled. Shared 30 bins; finite values only; out-of-range mass is not clipped.\n"
                "Windows were selected using real training cells within the published example locus; full-genome scores are unchanged.",
                fontsize=7, color="#555555")
    pd.concat(source_tables, ignore_index=True).to_csv(output / "A1_plot_histograms.csv", index=False)
    save_figure(figure, output, "A1_marginal_regions")


def plot_region_by_type(datasets, rows, methods, regions, output):
    detail = regions.loc[regions["单区域展示"].eq(1)].iloc[0]
    levels = detail["展示细胞类型"].split("|")
    names = ["真实数据", *methods]
    labels = {name: read_labels(datasets[name], rows[name]) for name in names}
    values = {name: dense_block(datasets[name], rows[name], [int(detail["原region索引"])]).ravel() for name in names}
    for name in names:
        if any(not np.any(labels[name] == level) for level in levels):
            raise ValueError(f"{name} 缺少清单指定的展示细胞类型，不静默混合或替换标签")
    edges = np.linspace(0, 1, 31)
    histograms = {(name, level): probability_histogram(values[name][labels[name] == level], edges)
                  for name in names for level in levels}
    upper = max(0.1, np.ceil(max(item[0].max() for item in histograms.values()) * 12) / 10)
    figure, axes = plt.subplots(len(methods), len(levels), figsize=(3.0 * len(levels), 1.9 * len(methods) + 1),
                                sharex=True, sharey=True, squeeze=False)
    figure.subplots_adjust(left=0.09, right=0.995, bottom=0.85 / figure.get_figheight(), top=1 - 1.0 / figure.get_figheight(), wspace=0.12, hspace=0.24)
    for i, method in enumerate(methods):
        for j, level in enumerate(levels):
            axis = axes[i, j]
            real_histogram, synthetic_histogram = histograms["真实数据", level], histograms[method, level]
            draw_marginal(axis, real_histogram, synthetic_histogram, method, edges)
            axis.text(0.02, 0.95, f"n: R={real_histogram[2]}, S={synthetic_histogram[2]}",
                      transform=axis.transAxes, fontsize=6.5, va="top", color="#555555")
            axis.set_ylim(0, upper)
            if i == 0:
                axis.set_title(level.replace(" CTX Glut", "\nCTX Glut"), fontsize=9, pad=8)
            if j == 0:
                axis.set_ylabel(f"{plot_name(method)}\nProbability per bin", color=plot_color(method), fontsize=8)
            if i == len(methods) - 1:
                axis.set_xlabel("Methylation level", fontsize=8)
    marginal_legend(figure, methods, 0.95)
    figure.suptitle(f"{detail['展示名称']} | {detail['原region']}\nSingle-region distributions by Subclass",
                   fontsize=11, weight="bold", y=1.01)
    figure.text(0.09, 0.015, "Rows: methods. Columns: saved cell types. R = evaluation real; S = synthetic; n counts finite observations.\n"
                "Shared 30 bins and y-axis; out-of-range values are reported separately.", fontsize=7, color="#555555")
    source_tables = [pd.DataFrame({"原region": detail["原region"], "细胞类型": level, "方法": name,
        "分箱左界": edges[:-1], "分箱右界": edges[1:], "概率": probability,
        "有限值数": count, "越界比例": outside})
        for (name, level), (probability, outside, count) in histograms.items()]
    pd.concat(source_tables, ignore_index=True).to_csv(output / "A1_type_histograms.csv", index=False)
    save_figure(figure, output, "A1_region_by_type")


def plot_a1(datasets, rows, methods, regions, output):
    plot_marginal_regions(datasets, rows, methods, regions, output)
    plot_region_by_type(datasets, rows, methods, regions, output)

## 执行与导出

校验完整区域轴及真实复制列；独立训练参照与真实留出细胞不得重叠。失败写入运行记录，读取对象在结束时关闭。

In [ ]:
def evaluate_experiment(config):
    methods = list(config["synthetic_h5ad"])
    paths = {"真实数据": resolve_path(config["real_h5ad"]),
             **{name: resolve_path(value) for name, value in config["synthetic_h5ad"].items()}}
    reference_path = resolve_path(config["reference_h5ad"]) if config["reference_h5ad"] else paths["真实数据"]
    pair_path = resolve_path(config["region_pairs_csv"]) if config["region_pairs_csv"] else None
    region_path = resolve_path(config["a1_regions_csv"]) if config.get("a1_regions_csv") else None
    output = resolve_path(config["output_dir"]) / EXPERIMENT
    if sys.platform != "win32":
        output.relative_to(Path("/home/sccasimp/data/xb/outputs/simscDNAm").resolve())
    all_paths = set(paths.values()) | {reference_path}
    if pair_path:
        all_paths.add(pair_path)
    if EXPERIMENT == "A1" and region_path:
        all_paths.add(region_path)
    if any(path.is_relative_to(output) for path in all_paths):
        raise ValueError("输出目录不能包含输入文件，以免覆盖输入；请选择独立的评估输出目录")
    for path in all_paths:
        if not path.is_file():
            raise FileNotFoundError(f"缺少输入文件：{path}")
    if len(set(paths.values())) != len(paths):
        raise ValueError("真实数据和各方法合成数据必须指向不同文件")
    records = [{"路径": str(path), "字节数": path.stat().st_size, "修改时间纳秒": path.stat().st_mtime_ns}
               for path in sorted(all_paths)]
    output.mkdir(parents=True, exist_ok=True)
    (output / "figures").mkdir(exist_ok=True)
    report = {"状态": "运行中", "标签字段": "Subclass", "最后复制区间数": 1000, "方法": methods,
              "参照模式": "独立训练真实参照" if reference_path != paths["真实数据"] else "评估真实数据拟合参照，非独立留出评估",
              "配置": config, "配置路径": str(CONFIG_PATH), "输入文件": records, "内核": sys.executable,
              "依赖版本": {name: importlib.metadata.version(name) for name in
                  ["numpy", "pandas", "scipy", "anndata", "scikit-learn", "umap-learn"]}}
    def write_report():
        (output / "run_summary.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    write_report()
    try:
        with ExitStack() as stack, threadpool_limits(limits=8):
            datasets = {}
            for name, path in paths.items():
                item = ad.read_h5ad(path, backed=None if config.get("read_in_memory", False) else "r")
                stack.callback(item.file.close)
                datasets[name] = item
            real = datasets["真实数据"]
            if reference_path == paths["真实数据"]:
                datasets["训练参照"] = real
            else:
                item = ad.read_h5ad(reference_path, backed=None if config.get("read_in_memory", False) else "r")
                stack.callback(item.file.close)
                datasets["训练参照"] = item
                if len(real.obs_names.intersection(item.obs_names)):
                    raise ValueError("独立训练参照与评估真实细胞 ID 重叠，请提供分离后的数据")
            for name, item in datasets.items():
                validate_region_axis(real, item)
                if not item.obs_names.is_unique:
                    raise ValueError(f"{name} 的细胞 ID 不唯一")
            pairs = region_pairs(real, pair_path=pair_path)
            validate_real_duplicates(real, pairs)
            if datasets["训练参照"] is not real:
                validate_real_duplicates(datasets["训练参照"], pairs)
            natural_count = real.n_vars - 1000
            count = real.n_obs if config["evaluation_cells"] is None else config["evaluation_cells"]
            rows = {name: sample_rows(item, count, config["seed"]) for name, item in datasets.items() if name != "训练参照"}
            rows["训练参照"] = np.arange(datasets["训练参照"].n_obs)
            labels = {name: read_labels(item, rows[name]) for name, item in datasets.items()}
            if EXPERIMENT == "A3" and 2 * count <= config["milisi_neighbors"]:
                raise ValueError("合并细胞数必须大于 miLISI 近邻数")
            pairs.to_csv(output / "region_pairs.csv", index=False)
            pd.concat([pd.DataFrame({"数据集": name, "原行索引": selected,
                "细胞ID": datasets[name].obs_names[selected], "细胞类型": labels[name]})
                for name, selected in rows.items()], ignore_index=True).to_csv(output / "selected_cells.csv", index=False)
            report.update({"评估细胞数": count, "真实参照细胞数": len(rows["训练参照"]), "原始区间数": natural_count,
                           "总区间数": real.n_vars, "A1_A3排除末尾复制列": True})
            write_report()
            print(f"输入校验通过：原始区间 {natural_count}，复制区间 1000，每种方法评估 {count} 个细胞。", flush=True)

            if region_path is None:
                raise ValueError("请通过 a1_regions_csv 提供有文献依据的展示清单；不再随机选择区域")
            plot_regions = load_plot_regions(real, natural_count, region_path, config.get("genome_assembly"))
            print("开始 A1：按真实类型加权的边缘指标。", flush=True)
            a1_rows, detail_tables = [], []
            for method in methods:
                summary, details = a1_metrics(real, datasets[method], rows["真实数据"], rows[method], natural_count)
                a1_rows.append({"方法": method, **summary})
                details.insert(0, "方法", method)
                detail_tables.append(details)
            a1 = pd.DataFrame(a1_rows)
            details = pd.concat(detail_tables, ignore_index=True)
            a1.to_csv(output / "A1_summary.csv", index=False)
            details.to_csv(output / "A1_type_details.csv", index=False)
            plot_a1(datasets, rows, methods, plot_regions, output)

            summary = a1
            diagnostics = []
            for name in ["真实数据", *methods]:
                for domain, start, stop in [("原始区间", 0, natural_count), ("复制区间", natural_count, real.n_vars)]:
                    valid, outside = 0, 0
                    for left in range(start, stop, 512):
                        block = dense_block(datasets[name], rows[name], slice(left, min(left + 512, stop)))
                        finite = np.isfinite(block)
                        valid += int(finite.sum())
                        outside += int((finite & ((block < 0) | (block > 1))).sum())
                    diagnostics.append({"数据集": name, "区间范围": domain, "有限观测数": valid,
                        "缺失比例": 1 - valid / (len(rows[name]) * (stop - start)),
                        "超出0到1比例": outside / valid if valid else np.nan})
            pd.DataFrame(diagnostics).to_csv(output / "input_diagnostics.csv", index=False)
            for record in records:
                path = Path(record["路径"])
                if path.stat().st_size != record["字节数"] or path.stat().st_mtime_ns != record["修改时间纳秒"]:
                    raise RuntimeError(f"评估期间输入文件发生变化：{path}")
            report.update({"状态": "完成", "输入文件大小及修改时间未变": True,
                           "输出目录": str(output), "输出文件": sorted(str(p.relative_to(output)) for p in output.rglob("*") if p.is_file())})
            write_report()
            display(summary)
            print(EXPERIMENT + " 评估完成，结果目录：", output)
            return summary
    except Exception as error:
        report.update({"状态": "失败", "错误": f"{type(error).__name__}: {error}"})
        write_report()
        raise


summary = evaluate_experiment(config)